# Cross-Benchmark Eval Overview

Loads transcripts from all eval benchmarks, compares pass rates, and
saves the combined DataFrame for downstream analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scan_utils import load_eval_logs

EVALS_ROOT = Path("../evals")

# Per-benchmark configuration.
# Keys: benchmark name used as display label.
# Values: dict with eval-log loading settings and optional validation info.
#
# validation_prefix: stripped from CSV filenames to derive column names.
#   For benchmarks with multiple prefixes (e.g. core_bench_easy_ and
#   core_bench_medium_), use the most common one and rename stragglers
#   via validation_renames.
# validation_renames: dict mapping load_validations output column names
#   to cleaner names (applied after prefix stripping).
BENCHMARKS = {
    "core_bench": {
        "eval_logs_dir": EVALS_ROOT / "core_bench" / "eval-logs",
        "scan_results_dir": EVALS_ROOT / "core_bench" / "scan-results",
        "validation_dir": EVALS_ROOT / "core_bench" / "validation",
        "validation_prefix": "core_bench_",
        "validation_renames": {
            "easy_oa1_JM": "oa1_JM",
            "easy_ob3_AH": "ob3_AH",
            "easy_ob3_JM": "ob3_JM",
            "easy_oh1_AH": "oh1_AH",
            "easy_oh1_JM": "oh1_JM",
            "easy_oh2_AH": "oh2_AH",
            "easy_oh2_JM": "oh2_JM",
            "medium_t2_JM": "t2_JM",
            "medium_t5_JM": "t5_JM",
        },
    },
    "compute_eval": {
        "eval_logs_dir": EVALS_ROOT / "compute_eval" / "eval-logs",
    },
    "cve_bench": {
        "eval_logs_dir": EVALS_ROOT / "cve_bench" / "eval-logs",
    },
    "gpqa_diamond": {
        "eval_logs_dir": EVALS_ROOT / "gpqa_diamond" / "eval-logs",
    },
    "hellaswag": {
        "eval_logs_dir": EVALS_ROOT / "hellaswag" / "eval-logs",
    },
    "kernelbench": {
        "eval_logs_dir": EVALS_ROOT / "kernelbench" / "eval-logs",
        "score_key": "correctness",
        "validation_dir": EVALS_ROOT / "kernelbench" / "validation",
        "validation_prefix": "kernelbench_",
        "scan_results_dir": EVALS_ROOT / "kernelbench" / "scan-results",
    },
    "litqa2": {
        "eval_logs_dir": EVALS_ROOT / "litqa2_astabench" / "eval-logs",
        "score_key": "is_correct",
    },
    "malt_dataset": {
        "eval_logs_dir": EVALS_ROOT / "malt_dataset" / "eval-logs",
        "score_key": "behavior_label",
        "success_fn": lambda v: v is not None and v != "bypass_constraints",
    },
    "mle_bench": {
        "eval_logs_dir": EVALS_ROOT / "mle_bench" / "eval-logs",
        "score_key": "above_median",
        "validation_dir": EVALS_ROOT / "mle_bench" / "validation",
        "validation_prefix": "mle_bench_",
        "scan_results_dir": EVALS_ROOT / "mle_bench" / "scan-results",
    },
    "mlrc_bench": {
        "eval_logs_dir": EVALS_ROOT / "mlrc_bench" / "eval-logs",
        "score_key": "error",
        "success_fn": lambda v: v is not None and float(v) == 0,
        "validation_dir": EVALS_ROOT / "mlrc_bench" / "validation",
        "validation_prefix": "mlrc_bench_",
        "scan_results_dir": EVALS_ROOT / "mlrc_bench" / "scan-results",
    },
    "super_astabench": {
        "eval_logs_dir": EVALS_ROOT / "super_astabench" / "eval-logs",
        "score_key": "output_match",
        "success_fn": lambda v: v is not None and float(v) == 1.0,
    },
    "swe_bench": {
        "eval_logs_dir": EVALS_ROOT / "swe_bench_verified" / "eval-logs",
        "scan_results_dir": EVALS_ROOT / "swe_bench_verified" / "scan-results",
        "validation_dir": EVALS_ROOT / "swe_bench_verified" / "validation",
        "validation_prefix": "swe_bench_",
    },
    "tau2": {
        "eval_logs_dir": EVALS_ROOT / "tau2" / "eval-logs",
        "validation_dir": EVALS_ROOT / "tau2" / "validation",
        "validation_prefix": "tau2_",
        "scan_results_dir": EVALS_ROOT / "tau2" / "scan-results",
    },
    "terminal_bench": {
        "eval_logs_dir": EVALS_ROOT / "terminal-bench-2.0" / "eval-logs",
        "validation_dir": EVALS_ROOT / "terminal-bench-2.0" / "validation",
        "validation_prefix": "",
        "validation_renames": {
            "oh1_terminal_bench_2": "oh1",
            "oh2_terminal_bench_2": "oh2",
            "t2_terminal_bench_2": "t2",
            "t5_terminal_bench_2": "t5",
        },
    },
    "truthfulqa": {
        "eval_logs_dir": EVALS_ROOT / "truthfulqa" / "eval-logs",
    },
    "winogrande": {
        "eval_logs_dir": EVALS_ROOT / "winogrande" / "eval-logs",
    },
    "xstest": {
        "eval_logs_dir": EVALS_ROOT / "xstest" / "eval-logs",
        "scan_results_dir": EVALS_ROOT / "xstest" / "scan-results",
    },
}

## 1. Load eval logs for all benchmarks

In [ ]:
all_logs: dict[str, pd.DataFrame] = {}

for name, cfg in BENCHMARKS.items():
    logs = load_eval_logs(
        cfg["eval_logs_dir"],
        score_key=cfg.get("score_key"),
        success_fn=cfg.get("success_fn"),
    )
    logs["benchmark"] = name
    all_logs[name] = logs
    n = len(logs)
    rate = logs["transcript_success"].mean()
    subsets = logs["eval_subset"].unique().tolist()
    print(f"{name:20s}  n={n:>4d}  pass={rate:.0%}  subsets: {subsets}")

combined_logs = pd.concat(all_logs.values(), ignore_index=True)
print(f"\nTotal transcripts: {len(combined_logs)}")

## 2. Overall pass rates by benchmark

In [ ]:
# Show only "default" (non-synthetic) transcripts for the headline pass rate.
default_logs = combined_logs[combined_logs["eval_subset"] == "default"]

pass_rates = (
    default_logs.groupby("benchmark")["transcript_success"]
    .agg(["mean", "size"])
    .rename(columns={"mean": "pass_rate", "size": "n"})
    .sort_values("pass_rate")
)

fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(pass_rates))))
pass_rates["pass_rate"].plot.barh(ax=ax, color="#4393c3")
ax.set_xlabel("Pass rate")
ax.set_title("Pass rate by benchmark (default / non-synthetic runs)")
ax.set_xlim(0, 1)
for i, (idx, row) in enumerate(pass_rates.iterrows()):
    ax.text(row["pass_rate"] + 0.01, i, f"{row['pass_rate']:.0%} (n={int(row['n'])})", va="center")
fig.tight_layout()
plt.show()

## 3. Pass rates by eval label (where applicable)

In [ ]:
labeled = combined_logs[combined_logs["eval_subset"] != "default"].copy()
if not labeled.empty:
    labeled["bench_label"] = labeled["benchmark"] + " / " + labeled["eval_subset"]
    label_rates = (
        labeled.groupby("bench_label")["transcript_success"]
        .agg(["mean", "size"])
        .rename(columns={"mean": "pass_rate", "size": "n"})
        .sort_values("pass_rate")
    )
    fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(label_rates))))
    label_rates["pass_rate"].plot.barh(ax=ax, color="#4393c3")
    ax.set_xlabel("Pass rate")
    ax.set_title("Pass rate by benchmark / eval label")
    ax.set_xlim(0, 1)
    for i, (idx, row) in enumerate(label_rates.iterrows()):
        ax.text(row["pass_rate"] + 0.01, i, f"{row['pass_rate']:.0%} (n={int(row['n'])})", va="center")
    fig.tight_layout()
    plt.show()
else:
    print("No benchmarks have eval labels.")

## 4. Summary table

In [ ]:
summary_table = (
    combined_logs.groupby(["benchmark", "eval_subset", "model"])
    .size()
    .rename("n_transcripts")
    .reset_index()
    .sort_values(["benchmark", "eval_subset", "model"])
)
display(summary_table.style.hide(axis="index").set_caption("Transcripts by benchmark, subset, and model"))


In [ ]:
display(combined_logs.head(10))

## 5. Save preprocessed data

In [ ]:
combined_logs.to_csv("preprocessed_eval_runs.csv", index=False)
print(f"Saved {len(combined_logs)} transcripts to preprocessed_eval_runs.csv")
